### 1.基于baseline(model A)论文配置训练环境

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install -q pyyaml

from pathlib import Path
import yaml

cfgp = Path("/content/drive/MyDrive/vehicle_reid_itsc2023-main/config/config.yaml")
d = yaml.safe_load(cfgp.read_text())
d["train_dir"] = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/dataset/VeRi/image_train"
d["query_dir"] = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/dataset/VeRi/image_query"
d["teste_dir"] = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/dataset/VeRi/image_test"
cfgp.write_text(yaml.safe_dump(d, sort_keys=False, allow_unicode=True))

In [ ]:
from pathlib import Path
import re, yaml

proj = Path("/content/drive/MyDrive/vehicle_reid_itsc2023-main")
cfg_path = proj/"config"/"config.yaml"
cfg = yaml.safe_load(cfg_path.read_text())

train_dir = Path(cfg["train_dir"])
assert train_dir.exists(), f"train_dir 不存在: {train_dir}"

vids, cams = set(), set()
for p in train_dir.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}:
        m_vid = re.match(r"^(\d+)_", p.name)    # VeRi 文件名风格
        m_cam = re.search(r"_c(\d+)", p.name, flags=re.I)
        vids.add(m_vid.group(1) if m_vid else p.parent.name)
        if m_cam: cams.add(m_cam.group(1))

if not cams:
    for d in train_dir.rglob("*"):
        if d.is_dir():
            m = re.search(r"c(\d+)", d.name, flags=re.I)
            if m: cams.add(m.group(1))

cfg["n_classes"] = int(len(vids))
cfg["n_cams"]    = int(max(1, len(cams)))
cfg.setdefault("LAI", True)
cfg.setdefault("n_groups", 4)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print("n_classes =", cfg['n_classes'], " | n_cams =", cfg['n_cams'])

### 2.复现论文得到模型

In [ ]:
%cd /content/drive/MyDrive/vehicle_reid_itsc2023-main
!python main.py --model_arch MBR_4G --config ./config/config_dark_1.yaml --batch_size 48

### 3.基于论文提供的teste.py文件，计算dataset中每个图像的AP值

In [ ]:
# ============================================================
# Notebook evaluation (paper-compatible) for VeRi-776
# - Reads query/gallery from config.yaml list files
# - ffs: per-branch L2 → concat; Euclidean distance (or re-ranking)
# - Junk removal: same pid & same cam excluded
# - Safe CPU loading of checkpoints; works on CPU or GPU
# - Optional: evaluate only top-N queries (Q_LIMIT)
# - Outputs overall mAP/CMC and per-query APs (and saves CSV)
# ============================================================

import os, sys, re, yaml, random, csv
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from contextlib import nullcontext

# ====== 1) 必改：工程与权重路径 ======
REPO_ROOT    = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"   # 你的仓库根目录
WEIGHTS_DIR  = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/5"                        # 里面包含 best_mAP.pt 与训练时保存的 config.yaml
WEIGHTS_NAME = "/content/best_mAP.pt"                                         # 或 last.pt / 你保存的名字
CONFIG_PATH  = os.path.join(WEIGHTS_DIR, "config.yaml")              # 默认用训练时保存的 config
# ====== 2) 只评测前 N 张 query；None=全量 ======
Q_LIMIT = 1600
# ====== 3) 是否做 k-reciprocal re-ranking（论文默认关）======
USE_RERANK = False
# ====== 4) 是否保存每张 AP（CSV）======
SAVE_CSV = True
CSV_PATH = "per_query_AP_topN.csv"   # 保存在当前工作目录

# ========= 环境准备 =========
assert os.path.isdir(REPO_ROOT), f"Repo not found: {REPO_ROOT}"
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from processor import get_model
from metrics.eval_reid import eval_func
from utils import re_ranking

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True

def set_seed(seed=0):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

# ========= 读取 config =========
assert os.path.isfile(CONFIG_PATH), f"config.yaml not found: {CONFIG_PATH}"
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

print("Loaded config:", CONFIG_PATH)
print("Dataset:", cfg['dataset'], "| model_arch:", cfg['model_arch'])

# ========= 从官方列表文件读取 Query/Gallery =========
query_dir   = cfg["query_dir"]
gallery_dir = cfg["teste_dir"]
q_list_file = cfg["query_list_file"]
g_list_file = cfg["gallery_list_file"]

assert os.path.isdir(query_dir),   f"query_dir not found: {query_dir}"
assert os.path.isdir(gallery_dir), f"teste_dir not found: {gallery_dir}"
assert os.path.isfile(q_list_file), f"query_list_file not found: {q_list_file}"
assert os.path.isfile(g_list_file), f"gallery_list_file not found: {g_list_file}"

ve_regex = re.compile(r"(\d+)_c(\d+)_", re.IGNORECASE)

def parse_pid_cam(name: str):
    m = ve_regex.search(name)
    if m:
        pid = int(m.group(1))
        cam = int(m.group(2)) - 1   # ★ 相机索引 0 基（c001→0）
        if cam < 0: cam = 0
    else:
        pid, cam = -1, 0
    view = 0  # VeRi 无显式视角标签，置 0
    return pid, cam, view

def read_list(list_file: str, root_dir: str):
    out = []
    with open(list_file, "r") as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            p = rel if os.path.isabs(rel) else os.path.join(root_dir, rel)
            if not os.path.exists(p):
                p2 = os.path.join(root_dir, os.path.basename(rel))
                if os.path.exists(p2):
                    p = p2
                else:
                    # print(f"[warn] not found: {p}")
                    continue
            out.append(os.path.abspath(p))
    if not out:
        raise RuntimeError(f"No entries read from {list_file}")
    return out

q_paths = read_list(q_list_file, query_dir)
g_paths = read_list(g_list_file, gallery_dir)
if Q_LIMIT is not None:
    q_paths = q_paths[:int(Q_LIMIT)]
print(f"#Query(subset)={len(q_paths)} | #Gallery(full)={len(g_paths)}")

# ========= Dataset/DataLoader =========
test_tf = transforms.Compose([
    transforms.Resize((cfg['y_length'], cfg['x_length']), antialias=True),
    transforms.Normalize(cfg['n_mean'], cfg['n_std']),
])

class ListDataset(Dataset):
    """返回 (tensor, pid, cam, view) —— 与论文 test_epoch 的接口一致"""
    def __init__(self, paths, transform):
        self.paths = paths
        self.tf = transform
        self.metas = [parse_pid_cam(os.path.basename(p)) for p in self.paths]
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        img = Image.open(p).convert("RGB")
        x = transforms.ToTensor()(img)
        x = self.tf(x)
        pid, cam, view = self.metas[i]
        return (
            x,
            torch.tensor(pid, dtype=torch.long),
            torch.tensor(cam, dtype=torch.long),
            torch.tensor(view, dtype=torch.long),
        )

dl_q = DataLoader(ListDataset(q_paths, test_tf),
                  batch_size=cfg['BATCH_SIZE'], shuffle=False,
                  num_workers=cfg['num_workers_teste'], pin_memory=True)
dl_g = DataLoader(ListDataset(g_paths, test_tf),
                  batch_size=cfg['BATCH_SIZE'], shuffle=False,
                  num_workers=cfg['num_workers_teste'], pin_memory=True)

# ========= 模型 & 安全加载权重（CPU/GPU 兼容）=========
from collections import OrderedDict

weights_path = os.path.join(WEIGHTS_DIR, WEIGHTS_NAME)
assert os.path.isfile(weights_path), f"weights not found: {weights_path}"

model = get_model(cfg, torch.device("cpu"))

def load_checkpoint_cpu_robust(path, model):
    ckpt = torch.load(path, map_location=torch.device('cpu'))  # ★ 强制CPU反序列化
    # A) 直接是 state_dict
    if isinstance(ckpt, dict) and all(isinstance(k, str) for k in ckpt.keys()) and any(hasattr(v, "shape") for v in ckpt.values()):
        sd = ckpt
    # B) {'state_dict': ...}
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    # C) 整模型（不推荐，但兜底）
    else:
        try:
            sd = ckpt.state_dict()
        except Exception as e:
            raise RuntimeError(f"Unsupported checkpoint format: {type(ckpt)}") from e
    # 去掉 DataParallel 的 'module.' 前缀
    new_sd = OrderedDict((k.replace('module.', ''), v) for k, v in sd.items())
    # 只加载 shape 匹配的参数
    msd = model.state_dict()
    matched = {k:v for k,v in new_sd.items() if (k in msd and v.shape == msd[k].shape)}
    msd.update(matched); model.load_state_dict(msd, strict=False)
    print(f"[safe-load] loaded={len(matched)}  skipped={len(new_sd)-len(matched)}")

load_checkpoint_cpu_robust(weights_path, model)
model = model.to(device).eval()

use_amp = bool(cfg.get('half_precision', True)) and (device.type == 'cuda')
print("Device:", device, "| AMP:", use_amp)

# ========= 评测：与论文 test_epoch 等价，并返回 per-query AP =========
@torch.no_grad()
def evaluate(model, device, dataloader_q, dataloader_g,
             use_amp=False, re_rank_flag=False, remove_junk=True, return_ap_list=True):
    model.eval()
    qf, gf = [], []
    q_camids, g_camids = [], []
    q_vids,   g_vids   = [], []

    amp_ctx = torch.cuda.amp.autocast if (use_amp and device.type=='cuda') else nullcontext

    # Query
    with amp_ctx():
        for image, q_id, cam_id, view_id in tqdm(dataloader_q, desc='Query infer (%)', ncols=120):
            image   = image.to(device, non_blocking=True)
            cam_id  = cam_id.to(device, non_blocking=True)
            view_id = view_id.to(device, non_blocking=True)
            out = model(image, cam_id, view_id)  # (_, _, ffs, _)
            assert isinstance(out, (list, tuple)) and len(out) >= 3, "Model output malformed"
            ffs = out[2]
            assert isinstance(ffs, (list, tuple)) and all(hasattr(f, 'shape') for f in ffs), "ffs must be a list of tensors"
            feat = torch.cat([F.normalize(f, dim=1) for f in ffs], dim=1)
            qf.append(feat); q_vids.append(q_id); q_camids.append(cam_id)

    # Gallery
    with amp_ctx():
        for image, g_id, cam_id, view_id in tqdm(dataloader_g, desc='Gallery infer (%)', ncols=120):
            image   = image.to(device, non_blocking=True)
            cam_id  = cam_id.to(device, non_blocking=True)
            view_id = view_id.to(device, non_blocking=True)
            out = model(image, cam_id, view_id)
            ffs = out[2]
            feat = torch.cat([F.normalize(f, dim=1) for f in ffs], dim=1)
            gf.append(feat); g_vids.append(g_id); g_camids.append(cam_id)

    qf = torch.cat(qf, dim=0)
    gf = torch.cat(gf, dim=0)

    # 距离矩阵（欧氏）或 re-ranking
    if re_rank_flag:
        distmat = re_ranking(qf, gf, k1=80, k2=16, lambda_value=0.3)
    else:
        m, n = qf.shape[0], gf.shape[0]
        distmat = (qf.pow(2).sum(dim=1, keepdim=True).expand(m, n) +
                   gf.pow(2).sum(dim=1, keepdim=True).expand(n, m).t())
        distmat.addmm_(qf, gf.t(), beta=1, alpha=-2)
        distmat = torch.sqrt(distmat).cpu().numpy()

    q_camids = torch.cat(q_camids, dim=0).cpu().numpy()
    g_camids = torch.cat(g_camids, dim=0).cpu().numpy()
    q_vids   = torch.cat(q_vids,   dim=0).cpu().numpy()
    g_vids   = torch.cat(g_vids,   dim=0).cpu().numpy()

    # 论文口径 mAP/CMC
    cmc, mAP = eval_func(distmat, q_vids, g_vids, q_camids, g_camids, remove_junk=remove_junk)

    if not return_ap_list:
        return cmc, mAP, None

    # 逐 query AP（与 junk 口径一致）
    def ap_from_sorted_hits(h):
        y = h.astype(np.int32)
        if y.sum() == 0: return 0.0
        cum = np.cumsum(y); ranks = np.arange(1, len(y)+1, dtype=np.float64)
        return float(((cum / ranks) * y).sum() / float(y.sum()))

    ap_list = []
    for i in range(distmat.shape[0]):
        d = distmat[i]; order = np.argsort(d)
        pid_i, cam_i = q_vids[i], q_camids[i]
        valid = ~((g_vids == pid_i) & (g_camids == cam_i)) if remove_junk else np.ones_like(g_vids, bool)
        matches = (g_vids == pid_i).astype(np.int32)
        keep = valid[order]; idx = order[keep]; hit = matches[idx]
        ap_list.append(ap_from_sorted_hits(hit))

    return cmc, mAP, ap_list

# ========= 执行评测 =========
cmc, mAP, ap_list = evaluate(model, device, dl_q, dl_g,
                             use_amp=use_amp,
                             re_rank_flag=USE_RERANK,
                             remove_junk=True,
                             return_ap_list=True)

print(f"\n[Top-{Q_LIMIT or 'ALL'} queries] mAP={mAP:.4f} | CMC@1={cmc[0]:.4f} | CMC@5={cmc[4] if len(cmc)>4 else float('nan'):.4f}")
if ap_list is not None:
    ap_arr = np.array(ap_list, dtype=float)
    print(f"Per-query AP mean={ap_arr.mean():.4f} | min={ap_arr.min():.4f} | max={ap_arr.max():.4f}")

# ========= （可选）保存逐张 AP 到 CSV =========
if SAVE_CSV and ap_list is not None:
    with open(CSV_PATH, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["idx_in_subset", "AP"])
        for i, ap in enumerate(ap_list):
            w.writerow([i, f"{ap:.6f}"])
    print(f"[saved] per-query AP -> {os.path.abspath(CSV_PATH)}")


### 4.统计AP值分布区间

In [ ]:
# ================== AP 分布统计（支持 idx_in_subset, AP 两列的 CSV） ==================
import os
import numpy as np
import pandas as pd

# 1) 修改这里为你的 CSV 路径（两列：idx_in_subset, AP）
CSV_PATH = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/per_query_AP_topN.csv"   # 例：你导出的 per-query AP CSV
OUT_BIN_CSV = "ap_bin_counts.csv"        # 输出分箱统计到这个文件

# 2) 读取 CSV（兼容有/无表头；兼容多余列但至少包含 AP 列）
def read_ap_csv(path):
    assert os.path.isfile(path), f"CSV 不存在: {path}"
    try:
        df = pd.read_csv(path)
        # 尝试常见列名
        if "AP" in df.columns and "idx_in_subset" in df.columns:
            ap = df["AP"].astype(float).values
            idx = df["idx_in_subset"].values
        elif df.shape[1] >= 2:
            # 没有表头或表头不标准：取第 1 列为 idx，第 2 列为 AP
            df0 = pd.read_csv(path, header=None)
            idx = df0.iloc[:,0].values
            ap  = df0.iloc[:,1].astype(float).values
        else:
            raise ValueError("CSV 列不足两列，或未找到 AP 列")
    except Exception as e:
        raise RuntimeError(f"读取 CSV 失败: {e}")
    return idx, ap

idx, ap = read_ap_csv(CSV_PATH)

# 3) 清洗 AP：去掉 NaN/inf，保留 [0,1] 范围内（可按需放宽）
mask = np.isfinite(ap) & (ap >= 0.0) & (ap <= 1.0)
bad = np.sum(~mask)
if bad > 0:
    print(f"[警告] 有 {bad} 个 AP 异常值（NaN/inf/越界），已剔除。")
ap = ap[mask]
idx = idx[mask]

n = len(ap)
assert n > 0, "有效 AP 为空。请检查 CSV 内容。"
print(f"有效样本数: {n}")

# 4) 总体统计
mean = float(np.mean(ap))
std  = float(np.std(ap, ddof=1)) if n > 1 else 0.0
ap_min, ap_max = float(np.min(ap)), float(np.max(ap))
q25, q50, q75 = np.percentile(ap, [25, 50, 75])

print("\n—— 总体统计 ——")
print(f"  mean={mean:.4f}, std={std:.4f}, min={ap_min:.4f}, median={q50:.4f}, max={ap_max:.4f}")
print(f"  Q1={q25:.4f}, Q2/Median={q50:.4f}, Q3={q75:.4f}")

# 5) 固定 10 桶分箱：[0.0,0.1)、…、[0.9,1.0]（最后一箱含右端点）
BIN_EDGES = np.linspace(0.0, 1.0, 11)
counts, edges = np.histogram(ap, bins=BIN_EDGES)
percents = counts / n * 100.0

print("\n—— AP 分箱统计（[左, 右)；最后一箱含右端点）——")
rows = []
for i in range(len(edges)-1):
    left, right = edges[i], edges[i+1]
    interval = f"[{left:.1f}, {right:.1f}]" if i == len(edges)-2 else f"[{left:.1f}, {right:.1f})"
    c, p = int(counts[i]), percents[i]
    rows.append((left, right, c, p))
    print(f"  {interval:12s}  count={c:5d}  {p:6.2f}%")


bin_df = pd.DataFrame(rows, columns=["bin_left", "bin_right", "count", "percent"])
bin_df.to_csv(OUT_BIN_CSV, index=False)
print(f"\n[已保存] 分箱统计 -> {os.path.abspath(OUT_BIN_CSV)}")

TOPK = min(10, n)
order = np.argsort(ap)
worst_idx = order[:TOPK]
best_idx  = order[::-1][:TOPK]

print(f"\n—— 最差 {TOPK} 个样本 ——")
for i in worst_idx:
    print(f"  idx={int(idx[i]):6d}  AP={ap[i]:.4f}")
print(f"\n—— 最好 {TOPK} 个样本 ——")
for i in best_idx:
    print(f"  idx={int(idx[i]):6d}  AP={ap[i]:.4f}")



### 5.绘制柱状图

In [ ]:
# Improved bar chart for AP distribution (matplotlib-only, single plot, no explicit colors)
# - Set CSV_PATH to your CSV: two columns (idx_in_subset, AP). Header is optional.
# - Produces a cleaner chart: larger fonts, y-grid, tight layout, percent labels.
# - Saves figure to /mnt/data/ap_distribution_hist_pretty.png and displays it.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== EDIT THIS PATH =====
CSV_PATH = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/per_query_AP_topN.csv"   # e.g., "/content/drive/MyDrive/per_query_AP.csv"
OUT_FIG = "/mnt/data/ap_distribution_hist_pretty.png"

def read_ap_csv(path):
    if not os.path.isfile(path):
        raise FileNotFoundError(f"CSV not found: {path}")
    try:
        df = pd.read_csv(path)
        if "AP" in df.columns:
            ap = df["AP"].astype(float).values
        else:
            df0 = pd.read_csv(path, header=None)
            if df0.shape[1] < 2:
                raise ValueError("CSV appears to have fewer than 2 columns.")
            ap = df0.iloc[:, 1].astype(float).values
    except Exception as e:
        raise RuntimeError(f"Failed to parse CSV: {e}")
    return ap

# Load and clean APs
ap = read_ap_csv(CSV_PATH)
mask = np.isfinite(ap) & (ap >= 0.0) & (ap <= 1.0)
ap = ap[mask]
n = len(ap)
if n == 0:
    raise ValueError("No valid AP values in [0,1].")

# Histogram (10 bins: [0.0,0.1),...,[0.9,1.0])
BIN_EDGES = np.linspace(0.0, 1.0, 11)
counts, edges = np.histogram(ap, bins=BIN_EDGES)
percents = counts / n * 100.0

# Build labels like [0.0,0.1), ... , [0.9,1.0]
labels = []
for i in range(len(edges)-1):
    left, right = edges[i], edges[i+1]
    lab = f"[{left:.1f}, {right:.1f})"
    if i == len(edges)-2:
        lab = f"[{left:.1f}, {right:.1f}]"
    labels.append(lab)

# Plot
plt.figure(figsize=(12, 6))
x = np.arange(len(counts))
bars = plt.bar(x, counts,color='orange')

# Improve readability
plt.xticks(x, labels, rotation=0, fontsize=10)
plt.yticks(fontsize=10)
plt.xlabel("AP Range", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.title("Per-Image AP Distribution", fontsize=14, pad=12)

# Add a light y-grid (default style, no explicit color)
plt.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.5)

# Annotate with count and percentage on top of bars
for xi, c, p in zip(x, counts, percents):
    # Offset a bit above the bar
    y = c + max(1, counts.max() * 0.02)
    plt.text(xi, y, f"{int(c)} ({p:.1f}%)", ha='center', va='bottom', fontsize=9)

# Add small headroom above the tallest bar
plt.ylim(0, max(counts) * 1.15 + 1)

plt.tight_layout()
plt.savefig(OUT_FIG, dpi=150, bbox_inches="tight")
plt.show()

OUT_FIG

### 6.将idx_in_subset转化为image_query中对应的图像编号，方便归纳用于训练缺陷模型

In [ ]:
import os, sys, re, yaml, pandas as pd

# ---- 路径与设置（按你当时评测时的设置保持一致）----
REPO_ROOT    = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"
WEIGHTS_DIR  = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/5"
CONFIG_PATH  = os.path.join(WEIGHTS_DIR, "config.yaml")
CSV_PATH     = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/per_query_AP_topN.csv"   # 你的 idx_in_subset, AP 的CSV
Q_LIMIT      = 1600   # 如果当时评测只跑了前10张，这里也填10；全量就填 None

# ---- 读取 config 拿到列表文件 ----
assert os.path.isfile(CONFIG_PATH), "config.yaml 不存在"
with open(CONFIG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

query_dir   = cfg["query_dir"]
q_list_file = cfg["query_list_file"]

# ---- 用与评测一致的方式读取 query 路径列表（不打乱）----
def read_list(list_file: str, root_dir: str):
    paths = []
    with open(list_file, "r") as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            p = rel if os.path.isabs(rel) else os.path.join(root_dir, rel)
            if not os.path.exists(p):
                # 备用：有些列表是相对路径但文件实际在根目录直接放文件名
                p2 = os.path.join(root_dir, os.path.basename(rel))
                if os.path.exists(p2):
                    p = p2
                else:
                    continue
            paths.append(os.path.abspath(p))
    return paths

q_paths = read_list(q_list_file, query_dir)
if Q_LIMIT is not None:
    q_paths = q_paths[:int(Q_LIMIT)]  # 与评测时一致

# ---- 读取你的 CSV，并映射 idx -> 文件名 ----
df = pd.read_csv(CSV_PATH)
# 如果你的CSV没有表头，使用：df = pd.read_csv(CSV_PATH, header=None, names=["idx_in_subset","AP"])
if "idx_in_subset" not in df.columns:
    # 兜底：假设第一列就是 idx
    df.columns = ["idx_in_subset","AP"]

def idx_to_name(i):
    i = int(i)
    if 0 <= i < len(q_paths):
        return os.path.basename(q_paths[i])  # 只要文件名；若想完整路径换成 q_paths[i]
    return "<out_of_range>"

df["query_image"] = df["idx_in_subset"].apply(idx_to_name)

# ---- 看看前几行 & 可保存新CSV ----
print(df.head(10))
out_csv = "per_query_AP_with_names.csv"
df.to_csv(out_csv, index=False)
print("已保存：", os.path.abspath(out_csv))


### 7.筛选出所有AP值小于0.8的图像

In [ ]:
# ==========================================
# 从 per_query_AP.csv 中筛出 AP < 0.8 的样本
# - 自动映射 idx_in_subset -> query 图片路径/文件名
# - 输出：ap_lt_0.8.csv（详细）、ap_lt_0.8.txt（仅路径）
# ==========================================
import os
import yaml
import pandas as pd

# ======== 修改为你的路径 ========
REPO_ROOT    = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"
WEIGHTS_DIR  = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/5"                    # 内含训练时保存的 config.yaml
CSV_PATH     = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/per_query_AP_with_names.csv"                           # 你的 per-query AP 文件

THRESH = 0.8
OUT_CSV = "ap_lt_0.8.csv"
OUT_TXT = "ap_lt_0.8.txt"

# ---------- 工具函数 ----------
def read_cfg(cfg_path: str):
    assert os.path.isfile(cfg_path), f"找不到 config.yaml: {cfg_path}"
    with open(cfg_path, "r") as f:
        return yaml.safe_load(f)

def read_list(list_file: str, root_dir: str):
    """按行读取列表文件，拼为绝对路径（兼容相对/仅文件名两种写法）"""
    assert os.path.isfile(list_file), f"query_list_file 不存在: {list_file}"
    assert os.path.isdir(root_dir),   f"query_dir 不存在: {root_dir}"
    paths = []
    with open(list_file, "r") as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            p = rel if os.path.isabs(rel) else os.path.join(root_dir, rel)
            if not os.path.exists(p):
                # 兜底：有些列表只写了文件名
                p2 = os.path.join(root_dir, os.path.basename(rel))
                if os.path.exists(p2):
                    p = p2
                else:
                    # 可以打印个告警；这里直接跳过
                    # print(f"[warn] not found: {p}")
                    continue
            paths.append(os.path.abspath(p))
    if not paths:
        raise RuntimeError(f"query 列表为空，请检查 {list_file} 与 {root_dir}")
    return paths

def read_ap_csv(csv_path: str):
    """读取 per_query_AP.csv（有/无表头都支持）→ DataFrame(idx_in_subset, AP)"""
    assert os.path.isfile(csv_path), f"CSV 不存在: {csv_path}"
    try:
        df = pd.read_csv(csv_path)
        if "AP" in df.columns and "idx_in_subset" in df.columns:
            return df[["idx_in_subset", "AP"]].copy()
        # 无规范表头：按两列读入
        df0 = pd.read_csv(csv_path, header=None, names=["idx_in_subset", "AP"])
        return df0[["idx_in_subset", "AP"]].copy()
    except Exception as e:
        raise RuntimeError(f"读取 CSV 失败: {e}")

# ---------- 读取 config & 构建 query 路径列表 ----------
CFG_PATH = os.path.join(WEIGHTS_DIR, "config.yaml")
cfg = read_cfg(CFG_PATH)

query_dir   = cfg["query_dir"]
q_list_file = cfg["query_list_file"]
q_paths = read_list(q_list_file, query_dir)

# ---------- 读取 per_query_AP.csv ----------
df = read_ap_csv(CSV_PATH)

# 规范化类型与范围
df["idx_in_subset"] = pd.to_numeric(df["idx_in_subset"], errors="coerce").astype("Int64")
df["AP"] = pd.to_numeric(df["AP"], errors="coerce")
df = df.dropna(subset=["idx_in_subset", "AP"]).copy()
df["idx_in_subset"] = df["idx_in_subset"].astype(int)
df = df[df["AP"].between(0, 1, inclusive="both")].copy()

# ---------- 映射 idx -> 路径/文件名 ----------
def idx_to_path(i: int):
    if 0 <= i < len(q_paths):
        return q_paths[i]
    return None

df["query_path"] = df["idx_in_subset"].apply(idx_to_path)
df["query_basename"] = df["query_path"].apply(lambda p: os.path.basename(p) if isinstance(p, str) else None)

# ---------- 筛选 AP < 阈值 并保存 ----------
bad = df[df["AP"] < THRESH].copy().reset_index(drop=True)
print(f"总样本数: {len(df)} | AP < {THRESH} 的样本数: {len(bad)}")

# 1) 详细 CSV
bad[["idx_in_subset", "AP", "query_basename", "query_path"]].to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# 2) 仅路径 TXT
with open(OUT_TXT, "w", encoding="utf-8") as f:
    for p in bad["query_path"]:
        if isinstance(p, str):
            f.write(p + "\n")

print("已保存：")
print(" -", os.path.abspath(OUT_CSV))
print(" -", os.path.abspath(OUT_TXT))


In [ ]:
# ==========================================
# 将 AP<阈值 的 query 图像集中到一个新文件夹（供训练用）
# - 读取 ap_lt_0.8.csv（支持：idx_in_subset, AP, [query_path|query_basename]）
# - 若无 query_path，则按 config.yaml + idx_in_subset 还原路径
# - 复制/链接到目标目录，并生成 name_train.txt 与 labels.csv
# ==========================================
import os
import re
import yaml
import shutil
import pandas as pd
from tqdm import tqdm

# ======== 修改为你的路径 ========
CSV_PATH    = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/ap_lt_0.8.csv"  # 你的筛选结果 CSV（含 AP<0.8）
REPO_ROOT   = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"  # 工程根目录
WEIGHTS_DIR = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/5"  # 内含训练时保存的 config.yaml
DEST_DIR    = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/dataset/MyData/low_ap_train"   # 输出目录（会自动创建）

# 阈值与拷贝策略
THRESH   = 0.8
LINK_MODE = 'copy'   # 'copy'（复制） | 'symlink'（软链接） | 'hardlink'（硬链接）

# ========== 基础检查 ==========
assert os.path.isfile(CSV_PATH), f"CSV 不存在: {CSV_PATH}"
CFG_PATH = os.path.join(WEIGHTS_DIR, "config.yaml")
assert os.path.isfile(CFG_PATH), f"找不到 config.yaml: {CFG_PATH}"
os.makedirs(DEST_DIR, exist_ok=True)

# ========== 读取 CSV ==========
# 兼容：有/无表头；若无表头则按两列/三列/四列兜底
def read_low_ap_csv(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    cols = [c.lower() for c in df.columns]
    # 规范列名（宽松匹配）
    rename_map = {}
    for c in df.columns:
        lc = c.lower()
        if lc == "idx_in_subset": rename_map[c] = "idx_in_subset"
        if lc == "ap":            rename_map[c] = "AP"
        if lc == "query_path":    rename_map[c] = "query_path"
        if lc == "query_basename":rename_map[c] = "query_basename"
    if rename_map:
        df = df.rename(columns=rename_map)

    # 无表头（或列名不规范）兜底：至少两列（idx, AP）
    if "AP" not in df.columns or "idx_in_subset" not in df.columns:
        df0 = pd.read_csv(csv_path, header=None)
        if df0.shape[1] < 2:
            raise ValueError("CSV 至少需要两列（idx_in_subset, AP）或包含表头。")
        # 尝试自动命名
        names = ["idx_in_subset", "AP"]
        if df0.shape[1] >= 3: names.append("query_basename")
        if df0.shape[1] >= 4: names.append("query_path")
        df0.columns = names + [f"extra_{i}" for i in range(df0.shape[1] - len(names))]
        df = df0

    # 清洗类型与范围
    df["idx_in_subset"] = pd.to_numeric(df["idx_in_subset"], errors="coerce").astype("Int64")
    df["AP"] = pd.to_numeric(df["AP"], errors="coerce")
    df = df.dropna(subset=["idx_in_subset", "AP"]).copy()
    df["idx_in_subset"] = df["idx_in_subset"].astype(int)
    df = df[df["AP"].between(0, 1, inclusive="both")].copy().reset_index(drop=True)
    return df

df = read_low_ap_csv(CSV_PATH)

# ========== 读取 config & 构建 query 路径列表 ==========
with open(CFG_PATH, "r") as f:
    cfg = yaml.safe_load(f)

def read_list(list_file: str, root_dir: str):
    paths = []
    with open(list_file, "r") as f:
        for line in f:
            rel = line.strip()
            if not rel:
                continue
            p = rel if os.path.isabs(rel) else os.path.join(root_dir, rel)
            if not os.path.exists(p):
                # 兜底：只写了文件名的情况
                p2 = os.path.join(root_dir, os.path.basename(rel))
                if os.path.exists(p2):
                    p = p2
                else:
                    # 打印告警也可以；这里直接跳过
                    continue
            paths.append(os.path.abspath(p))
    return paths

q_paths_all = read_list(cfg["query_list_file"], cfg["query_dir"])
assert len(q_paths_all) > 0, "query 列表为空，请检查 config.yaml 的 query_list_file / query_dir"

# ========== 若无 query_path，则按 idx_in_subset 还原 ==========
if "query_path" not in df.columns or df["query_path"].isna().all():
    def idx_to_path(i: int):
        if 0 <= i < len(q_paths_all):
            return q_paths_all[i]
        return None
    df["query_path"] = df["idx_in_subset"].apply(idx_to_path)
else:
    # 规范化为绝对路径（若列里提供的是 basename）
    def canon_path(x):
        if isinstance(x, str) and os.path.isabs(x) and os.path.exists(x):
            return os.path.abspath(x)
        if isinstance(x, str) and os.path.exists(x):
            return os.path.abspath(x)
        # 尝试用 query_dir 補全 basename
        try_path = os.path.join(cfg["query_dir"], os.path.basename(str(x)))
        return os.path.abspath(try_path) if os.path.exists(try_path) else None
    df["query_path"] = df["query_path"].apply(canon_path)

# 仅保留 AP < 阈值 且路径有效
df_low = df[(df["AP"] < THRESH) & df["query_path"].notna()].copy().reset_index(drop=True)
paths = df_low["query_path"].astype(str).tolist()
print(f"AP<{THRESH} 有效图像数：{len(paths)}")

# ========== VeRi 命名解析（获得 pid/cam）==========
ve_regex = re.compile(r"(\d+)_c(\d+)_", re.IGNORECASE)
def parse_pid_cam_from_name(basename: str):
    m = ve_regex.search(basename)
    if m:
        pid = int(m.group(1))
        cam = int(m.group(2))
    else:
        pid, cam = -1, -1
    return pid, cam

# ========== 拷贝/链接到目标目录 ==========
copied = []
missing = []
for src in tqdm(paths, desc="Collecting images"):
    if not os.path.isfile(src):
        missing.append(src)
        continue
    base = os.path.basename(src)
    dst = os.path.join(DEST_DIR, base)

    # 如果重名，避免覆盖：加编号
    if os.path.exists(dst):
        stem, ext = os.path.splitext(base)
        k = 1
        while True:
            new_base = f"{stem}_{k}{ext}"
            dst = os.path.join(DEST_DIR, new_base)
            if not os.path.exists(dst):
                base = new_base
                break
            k += 1

    try:
        if LINK_MODE == 'copy':
            shutil.copy2(src, dst)
        elif LINK_MODE == 'symlink':
            os.symlink(src, dst)  # 某些平台可能需要权限；失败就改用 copy
        elif LINK_MODE == 'hardlink':
            os.link(src, dst)     # 不同文件系统可能不支持；失败就改用 copy
        else:
            raise ValueError("LINK_MODE must be one of: 'copy', 'symlink', 'hardlink'")
        copied.append(base)
    except Exception as e:
        # 兜底改为复制
        shutil.copy2(src, dst)
        copied.append(base)

print(f"完成：成功 {len(copied)} 张；缺失 {len(missing)} 张。目标目录：{DEST_DIR}")

# ========== 生成训练清单与标签 ==========
# 1) name_train.txt：每行一个文件名（相对 DEST_DIR）
list_txt = os.path.join(DEST_DIR, "name_train.txt")
with open(list_txt, "w", encoding="utf-8") as f:
    for name in copied:
        f.write(name + "\n")
print("已保存列表：", list_txt)

# 2) labels.csv：filename, pid, cam
rows = []
for name in copied:
    pid, cam = parse_pid_cam_from_name(name)
    rows.append({"filename": name, "pid": pid, "cam": cam})
labels_csv = os.path.join(DEST_DIR, "labels.csv")
pd.DataFrame(rows).to_csv(labels_csv, index=False, encoding="utf-8-sig")
print("已保存标签：", labels_csv)


### 8.基于论文提供的参数修改配置，增大learning rate，triplet_margin等参数用于训练缺陷模型

In [ ]:
%cd /content/drive/MyDrive/vehicle_reid_itsc2023-main
!python main.py --model_arch MBR_4G --config ./config/config_dark_2.yaml --batch_size 32
# 显存不够可降到 24 或 16，且要能被 NUM_INSTANCES(=4)整除


### 9.通过best stragety将两个模型融合在一起，用多模型融合的方式优化论文模型，提升map

In [ ]:
# ============================================================
# Dual-config, dual-model evaluation (paper-compatible)
# - 每个模型按照自己的 config 构建与预处理
# - 评测使用同一批 query/gallery（默认以 A 的 list 为准；可切换为交集模式）
# - 输出：A/B 的 mAP/CMC；逐张 AP；逐张 max(AP) 及其来源；保存 CSV
# ============================================================
import os, sys, re, yaml, random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from collections import OrderedDict
from contextlib import nullcontext

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ========= 必改：两个模型与其 config =========
# 模型 A（比如“原基线”）
REPO_ROOT_A      = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"  # 仓库根目录
CONFIG_A_PATH   = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/5/config.yaml"      # A 的 config.yaml
WEIGHTS_A_PATH  = "/content/best_mAP.pt"    # A 的权重

# 模型 B（比如“黑暗模型”）
REPO_ROOT_B     = "/content/drive/MyDrive/vehicle_reid_itsc2023-main"
CONFIG_B_PATH   = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/9/config.yaml"
WEIGHTS_B_PATH  = "/content/drive/MyDrive/vehicle_reid_itsc2023-main/logs/Veri776/MBR_4G/9/best_mAP.pt"


# 评测集选择策略：
USE_LIST = "A"     # "A"：用 A 的 query_list_file / gallery_list_file
                   # "B"：用 B 的列表
                   # "INTERSECT"：A 与 B 两边列表文件的“交集（按文件名或相对路径匹配）”
Q_LIMIT  = None    # 只评测前 N 张；None=全量
USE_RERANK = False # 是否启用 re-ranking（论文默认关闭）
OUT_CSV  = "per_query_AP_dual_by_config.csv"

# ========= 把两个 repo 都加到 sys.path（如相同仓库可只加一个）=========
for root in {REPO_ROOT_A, REPO_ROOT_B}:
    assert os.path.isdir(root), f"Repo not found: {root}"
    if root not in sys.path:
        sys.path.insert(0, root)

from processor import get_model
from metrics.eval_reid import eval_func
from utils import re_ranking

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def set_seed(seed=0):
    torch.manual_seed(seed); torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(0)

# ========= 读 config =========
def load_cfg(path):
    assert os.path.isfile(path), f"config.yaml not found: {path}"
    with open(path, "r") as f:
        return yaml.safe_load(f)

cfgA = load_cfg(CONFIG_A_PATH)
cfgB = load_cfg(CONFIG_B_PATH)

# ========= 读取列表文件（VeRi-776）=========
ve_regex = re.compile(r"(\d+)_c(\d+)_", re.IGNORECASE)
def parse_pid_cam(name: str):
    m = ve_regex.search(name)
    if m:
        pid = int(m.group(1)); cam = int(m.group(2)) - 1
        cam = 0 if cam < 0 else cam
    else:
        pid, cam = -1, 0
    view = 0
    return pid, cam, view

def read_list(list_file: str, root_dir: str):
    out = []
    with open(list_file, "r") as f:
        for line in f:
            rel = line.strip()
            if not rel: continue
            p = rel if os.path.isabs(rel) else os.path.join(root_dir, rel)
            if not os.path.exists(p):
                p2 = os.path.join(root_dir, os.path.basename(rel))
                if os.path.exists(p2): p = p2
                else: continue
            out.append(os.path.abspath(p))
    if not out:
        raise RuntimeError(f"No entries read from {list_file}")
    return out

def get_paths_from_cfg(cfg):
    qdir, gdir = cfg["query_dir"], cfg["teste_dir"]
    qlist, glist = cfg["query_list_file"], cfg["gallery_list_file"]
    assert os.path.isdir(qdir) and os.path.isdir(gdir)
    assert os.path.isfile(qlist) and os.path.isfile(glist)
    return read_list(qlist, qdir), read_list(glist, gdir)

qA, gA = get_paths_from_cfg(cfgA)
qB, gB = get_paths_from_cfg(cfgB)

if USE_LIST == "A":
    q_paths, g_paths = qA, gA
elif USE_LIST == "B":
    q_paths, g_paths = qB, gB
elif USE_LIST == "INTERSECT":
    # 交集（按文件名匹配），并保持与 A 的顺序
    set_qB = {os.path.basename(p) for p in qB}
    q_paths = [p for p in qA if os.path.basename(p) in set_qB]
    set_gB = {os.path.basename(p) for p in gB}
    g_paths = [p for p in gA if os.path.basename(p) in set_gB]
else:
    raise ValueError("USE_LIST must be 'A', 'B' or 'INTERSECT'")

if Q_LIMIT is not None:
    q_paths = q_paths[:int(Q_LIMIT)]
print(f"#Query={len(q_paths)} | #Gallery={len(g_paths)} | USE_LIST={USE_LIST}")

# ========= 为每个模型按各自 config 构造 transform =========
def build_test_transform(cfg):
    return transforms.Compose([
        transforms.Resize((cfg['y_length'], cfg['x_length']), antialias=True),
        transforms.Normalize(cfg['n_mean'], cfg['n_std']),
    ])

tfA = build_test_transform(cfgA)
tfB = build_test_transform(cfgB)

# ========= Dataset，支持“共享路径、独立 transform” =========
class ListDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.tf = transform
        self.metas = [parse_pid_cam(os.path.basename(p)) for p in self.paths]
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        img = Image.open(p).convert("RGB")
        x = transforms.ToTensor()(img)
        x = self.tf(x)
        pid, cam, view = self.metas[i]
        return (
            x,
            torch.tensor(pid, dtype=torch.long),
            torch.tensor(cam, dtype=torch.long),
            torch.tensor(view, dtype=torch.long),
        )

# 为 A/B 构建各自的 DataLoader（共享同一组路径）
def build_loaders_for_cfg(cfg, tf, q_paths, g_paths):
    bs = int(cfg['BATCH_SIZE'])
    nw = int(cfg['num_workers_teste'])
    dl_q = DataLoader(ListDataset(q_paths, tf), batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    dl_g = DataLoader(ListDataset(g_paths, tf), batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True)
    return dl_q, dl_g

dl_qA, dl_gA = build_loaders_for_cfg(cfgA, tfA, q_paths, g_paths)
dl_qB, dl_gB = build_loaders_for_cfg(cfgB, tfB, q_paths, g_paths)

# ========= 模型安全加载（各自 config）=========
def safe_load_to_model(weights_path, model):
    ckpt = torch.load(weights_path, map_location=torch.device('cpu'))
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        sd = ckpt['state_dict']
    elif isinstance(ckpt, dict) and any(hasattr(v, "shape") for v in ckpt.values()):
        sd = ckpt
    else:
        try: sd = ckpt.state_dict()
        except Exception as e: raise RuntimeError(f"Unsupported checkpoint format: {type(ckpt)}") from e
    new_sd = OrderedDict((k.replace('module.', ''), v) for k, v in sd.items())
    msd = model.state_dict()
    matched = {k: v for k, v in new_sd.items() if (k in msd and v.shape == msd[k].shape)}
    msd.update(matched); model.load_state_dict(msd, strict=False)
    print(f"[safe-load] {os.path.basename(weights_path)}  loaded={len(matched)}  skipped={len(new_sd)-len(matched)}")

def build_model_from_cfg(cfg, weights_path):
    m = get_model(cfg, torch.device("cpu"))
    safe_load_to_model(weights_path, m)
    return m.to(device).eval()

modelA = build_model_from_cfg(cfgA, WEIGHTS_A_PATH)
modelB = build_model_from_cfg(cfgB, WEIGHTS_B_PATH)

use_amp_A = bool(cfgA.get('half_precision', False)) and (device.type == 'cuda')
use_amp_B = bool(cfgB.get('half_precision', False)) and (device.type == 'cuda')
ampA = torch.cuda.amp.autocast if use_amp_A else nullcontext
ampB = torch.cuda.amp.autocast if use_amp_B else nullcontext

# ========= 抽特征（按各自 config 的 transform / DataLoader）=========
@torch.no_grad()
def extract_features(model, dl_q, dl_g, amp_ctx):
    qf, gf = [], []
    q_vids, q_camids = [], []
    g_vids, g_camids = [], []
    with amp_ctx():
        for image, pid, cam, view in tqdm(dl_q, desc='Query infer', ncols=120):
            image, cam, view = image.to(device), cam.to(device), view.to(device)
            out = model(image, cam, view)  # (_, _, ffs, _)
            ffs = out[2]
            feat = torch.cat([F.normalize(f, dim=1) for f in ffs], dim=1)
            qf.append(feat); q_vids.append(pid); q_camids.append(cam)
    with amp_ctx():
        for image, pid, cam, view in tqdm(dl_g, desc='Gallery infer', ncols=120):
            image, cam, view = image.to(device), cam.to(device), view.to(device)
            out = model(image, cam, view)
            ffs = out[2]
            feat = torch.cat([F.normalize(f, dim=1) for f in ffs], dim=1)
            gf.append(feat); g_vids.append(pid); g_camids.append(cam)
    qf = torch.cat(qf, dim=0); gf = torch.cat(gf, dim=0)
    q_vids = torch.cat(q_vids, dim=0).cpu().numpy()
    q_camids = torch.cat(q_camids, dim=0).cpu().numpy()
    g_vids = torch.cat(g_vids, dim=0).cpu().numpy()
    g_camids = torch.cat(g_camids, dim=0).cpu().numpy()
    return qf, gf, q_vids, q_camids, g_vids, g_camids

print("\n==> Model A extracting…")
qfA, gfA, q_vids, q_camids, g_vids, g_camids = extract_features(modelA, dl_qA, dl_gA, ampA)
print("==> Model B extracting…")
qfB, gfB, *_ = extract_features(modelB, dl_qB, dl_gB, ampB)

# ========= 计算距离、mAP/CMC、逐张 AP =========
def compute_distmat(qf, gf):
    if USE_RERANK:
        return re_ranking(qf, gf, k1=80, k2=16, lambda_value=0.3)
    m, n = qf.shape[0], gf.shape[0]
    dist = (qf.pow(2).sum(dim=1, keepdim=True).expand(m, n) +
            gf.pow(2).sum(dim=1, keepdim=True).expand(n, m).t())
    dist.addmm_(qf, gf.t(), beta=1, alpha=-2)
    return torch.sqrt(dist).cpu().numpy()

def per_query_ap_list(distmat, q_vids, g_vids, q_camids, g_camids, remove_junk=True):
    def ap_from_sorted_hits(h):
        y = h.astype(np.int32)
        if y.sum() == 0: return 0.0
        cum = np.cumsum(y); ranks = np.arange(1, len(y)+1, dtype=np.float64)
        return float(((cum / ranks) * y).sum() / float(y.sum()))
    aps = []
    for i in range(distmat.shape[0]):
        d = distmat[i]; order = np.argsort(d)
        pid_i, cam_i = q_vids[i], q_camids[i]
        valid = ~((g_vids == pid_i) & (g_camids == cam_i)) if remove_junk else np.ones_like(g_vids, bool)
        matches = (g_vids == pid_i).astype(np.int32)
        hit = matches[order][valid[order]]
        aps.append(ap_from_sorted_hits(hit))
    return np.array(aps, dtype=float)

print("\n==> Metrics…")
distA = compute_distmat(qfA, gfA)
distB = compute_distmat(qfB, gfB)

cmcA, mAP_A = eval_func(distA, q_vids, g_vids, q_camids, g_camids, remove_junk=True)
cmcB, mAP_B = eval_func(distB, q_vids, g_vids, q_camids, g_camids, remove_junk=True)

apA = per_query_ap_list(distA, q_vids, g_vids, q_camids, g_camids, remove_junk=True)
apB = per_query_ap_list(distB, q_vids, g_vids, q_camids, g_camids, remove_junk=True)

ap_max = np.maximum(apA, apB)
chosen = np.where(apA >= apB, "A", "B")

print(f"\nModel A : mAP={mAP_A:.4f} | CMC@1={cmcA[0]:.4f} | CMC@5={cmcA[4] if len(cmcA)>4 else float('nan'):.4f}")
print(f"Model B : mAP={mAP_B:.4f} | CMC@1={cmcB[0]:.4f} | CMC@5={cmcB[4] if len(cmcB)>4 else float('nan'):.4f}")
print(f"Fusion (per-query max AP): mean(AP_max)={ap_max.mean():.4f} | min={ap_max.min():.4f} | max={ap_max.max():.4f}")

# ========= 保存 CSV =========
rows = []
for i, qp in enumerate(q_paths):
    rows.append({
        "idx_in_subset": i,
        "query_name": os.path.basename(qp),
        "AP_modelA": float(apA[i]),
        "AP_modelB": float(apB[i]),
        "AP_max": float(ap_max[i]),
        "chosen_model": chosen[i],  # A or B
    })
pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", os.path.abspath(OUT_CSV))
